# NB02 — Metadata harmonisation and quality control
Builds one harmonised sample sheet per cohort with the columns
`sample_id, cohort, group, idh_status, primary_recurrent, age, sex, os_time_days, os_event, treatment fields,
included, exclusion_reason`, applying the pre-specified population (adult, primary, IDH-wildtype GBM) and logging
**every** exclusion with a reason. Also screens for duplicate patients across cohorts.

Rules fixed in advance:
* control labels in microarray series are mapped by explicit keyword rules and **audited** (raw label -> group table is saved);
* IDH-wildtype in TCGA requires the patient to appear in the mutation file with no IDH1/IDH2 coding variant
  (absence from the file means *unknown*, never wildtype);
* legacy array cohorts (GSE4290, GSE50161) have no IDH annotation, so they are used only for the control-type
  analysis (H2) and flagged `idh_status = not_annotated`.

In [2]:
import os, sys
ROOT = os.environ.get("GBM_ROOT", "/content/drive/MyDrive/GBM_MultiCohort_Research")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab; GBM_ROOT =", ROOT)
sys.path.insert(0, os.path.join(ROOT, "00_Project_Config"))
from gbm_common import *
RAW = P("raw")
exclusions = []   # (cohort, sample_id, reason)
print("SMOKE =", SMOKE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SMOKE = False


In [4]:
import os, sys, re, gzip
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path(os.environ.get("GBM_ROOT", "/content/drive/MyDrive/GBM_MultiCohort_Research"))
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
sys.path.insert(0, str(ROOT / '00_Project_Config'))
from gbm_common import *
RAW = P('raw')
exclusions = []
P('meta', 'placeholder.txt').unlink(missing_ok=True)  # ensure metadata directory exists
P('qc', 'placeholder.txt').unlink(missing_ok=True)

def optional_table(path):
    if not path.exists():
        print(f'WARNING missing optional file: {path}')
        return None
    return pd.read_csv(path, sep='\t', compression='infer', low_memory=False)

def norm_patient(x):
    return x.astype('string').str.upper().str.extract(r'(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})', expand=False)

def norm_sample(x):
    return x.astype('string').str.upper().str.extract(r'(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}-[0-9]{2})', expand=False)

print('SMOKE =', SMOKE, '| ROOT =', ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SMOKE = False | ROOT = /content/drive/MyDrive/GBM_MultiCohort_Research


In [7]:
import pandas as pd

path = RAW / "GEO" / "GSE4290_series_matrix.txt.gz"
m, _ = read_geo_series_matrix(path)

print("Available metadata columns:")
print(m.columns.tolist())

for col in m.columns:
    if any(x in col.lower() for x in [
        "character", "hist", "grade", "diagnos",
        "tumor", "tumour", "title", "source"
    ]):
        print(f"\nCOLUMN: {col}")
        print(m[col].value_counts(dropna=False).head(30))

Available metadata columns:
['title', 'geo_accession', 'status', 'submission_date', 'last_update_date', 'type', 'channel_count', 'source_name_ch1', 'organism_ch1', 'molecule_ch1', 'extract_protocol_ch1', 'label_ch1', 'label_protocol_ch1', 'taxid_ch1', 'hyb_protocol', 'scan_protocol', 'description', 'data_processing', 'platform_id', 'contact_name', 'contact_email', 'contact_phone', 'contact_fax', 'contact_department', 'contact_institute', 'contact_address', 'contact_city', 'contact_state', 'contact_zip/postal_code', 'contact_country', 'supplementary_file', 'data_row_count', 'relation', 'histopathological_diagnostic']

COLUMN: title
title
HF0017_U133P2    1
HF0024_U133P2    1
HF0026_U133P2    1
HF0031_U133P2    1
HF0050_U133P2    1
HF0066_U133P2    1
HF0087_U133P2    1
HF0088_U133P2    1
HF0089_U133P2    1
HF0108_U133P2    1
HF0120_U133P2    1
HF0131_U133P2    1
HF0137_U133P2    1
HF0138_U133P2    1
HF0141_U133P2    1
HF0142_U133P2    1
HF0151_U133P2    1
HF0152_U133P2    1
HF0163_U133P2

In [8]:
# ---- A. GEO microarray cohorts: corrected classification ----

import re
import numpy as np
import pandas as pd

LABEL_RULES = {
    "control": [
        "non-tumor", "non tumor", "nontumor",
        "normal", "control", "epilep"
    ],
    "GBM": [
        "glioblastoma", "gbm",
        "grade 4", "grade iv",
        "who iv", "who grade iv"
    ]
}

HIST_COLS = [
    "histology", "diagnosis", "disease_state",
    "disease", "tissue", "source_name_ch1",
    "title"
]

def classify_geo_samples(metadata):
    """Classify only explicitly supported GBM/control labels."""

    cols = [
        c for c in metadata.columns
        if (
            c in HIST_COLS
            or "characteristics" in c.lower()
            or "histolog" in c.lower()
            or "grade" in c.lower()
            or "diagnos" in c.lower()
        )
    ]

    if not cols:
        raise ValueError(
            "No usable phenotype columns found: "
            + str(metadata.columns.tolist())
        )

    combined = (
        metadata[cols]
        .fillna("")
        .astype(str)
        .agg(" | ".join, axis=1)
        .str.lower()
    )

    gbm_pattern = (
        r"\bglioblastoma\b|\bgbm\b|"
        r"\bgrade\s*(?:iv|4)\b|"
        r"\bwho\s*(?:grade\s*)?(?:iv|4)\b"
    )

    control_pattern = (
        r"non[\s-]?tumou?r|"
        r"\bnormal\b|\bcontrol\b|epilep"
    )

    other_grade_pattern = (
        r"\bgrade\s*(?:i|ii|iii|1|2|3)\b|"
        r"\bwho\s*(?:grade\s*)?(?:i|ii|iii|1|2|3)\b"
    )

    gbm = combined.str.contains(
        gbm_pattern, regex=True, na=False
    )

    control = combined.str.contains(
        control_pattern, regex=True, na=False
    )

    other_grade = combined.str.contains(
        other_grade_pattern, regex=True, na=False
    )

    # Do not classify contradictory metadata automatically.
    ambiguous = (
        (gbm & control)
        | (gbm & other_grade)
    )

    group = pd.Series(
        "other", index=metadata.index, dtype="object"
    )

    group.loc[control & ~gbm] = "control"
    group.loc[gbm & ~control & ~other_grade] = "GBM"
    group.loc[ambiguous] = "ambiguous"

    return group, combined


array_sheets = {}

for acc in CFG["geo_microarray"]:

    path = RAW / "GEO" / f"{acc}_series_matrix.txt.gz"
    require_files([path], "Run NB01 first.")

    m, _ = read_geo_series_matrix(path)

    group, combined_labels = classify_geo_samples(m)

    audit = pd.DataFrame({
        "sample_id": m.index,
        "combined_metadata": combined_labels.values,
        "assigned_group": group.values
    })

    audit_path = P("meta", f"{acc}_label_mapping_audit.csv")
    audit.to_csv(audit_path, index=False)

    sheet = pd.DataFrame({
        "sample_id": m.index,
        "cohort": acc,
        "group": group.values,
        "raw_label": combined_labels.values,
        "idh_status": "not_annotated",
        "primary_recurrent": "unknown",
        "age": np.nan,
        "sex": "unknown",
        "os_time_days": np.nan,
        "os_event": np.nan
    })

    sheet["included"] = sheet["group"].isin(
        ["GBM", "control"]
    )

    sheet["exclusion_reason"] = np.where(
        sheet["included"],
        "",
        "non-GBM or insufficient/ambiguous histology"
    )

    for sid in sheet.loc[
        ~sheet["included"], "sample_id"
    ]:
        exclusions.append(
            (acc, sid, "non-GBM or ambiguous histology")
        )

    array_sheets[acc] = sheet

    n_gbm = int((sheet["group"] == "GBM").sum())
    n_ctl = int((sheet["group"] == "control").sum())

    print(f"\n{acc} classification:")
    print(sheet["group"].value_counts())
    print("GBM:", n_gbm)
    print("Control:", n_ctl)
    print("Audit:", audit_path)

    min_n = CFG["preprocessing"]["min_group_n"]

    if n_gbm < min_n or n_ctl < min_n:
        print(
            f"WARNING: {acc} has insufficient verified "
            f"GBM/control samples for differential expression. "
            "Review the audit before using this cohort."
        )


GSE4290 classification:
group
other      80
GBM        77
control    23
Name: count, dtype: int64
GBM: 77
Control: 23
Audit: /content/drive/MyDrive/GBM_MultiCohort_Research/03_Metadata/GSE4290_label_mapping_audit.csv

GSE50161 classification:
group
other      83
GBM        34
control    13
Name: count, dtype: int64
GBM: 34
Control: 13
Audit: /content/drive/MyDrive/GBM_MultiCohort_Research/03_Metadata/GSE50161_label_mapping_audit.csv


In [11]:
# GSE4290 metadata diagnostic

path = RAW / "GEO" / "GSE4290_series_matrix.txt.gz"
m, _ = read_geo_series_matrix(path)

print("=== ALL METADATA COLUMNS ===")
print(m.columns.tolist())

print("\n=== HISTOLOGY / GRADE / DIAGNOSIS ===")

for col in m.columns:
    if any(word in col.lower() for word in [
        "character", "hist", "grade",
        "diagnos", "title", "source"
    ]):
        print(f"\nCOLUMN: {col}")
        print(m[col].value_counts(dropna=False).head(40).to_string())

print("\n=== SAMPLE EXAMPLES ===")
print(m.head(10).to_string())

=== ALL METADATA COLUMNS ===
['title', 'geo_accession', 'status', 'submission_date', 'last_update_date', 'type', 'channel_count', 'source_name_ch1', 'organism_ch1', 'molecule_ch1', 'extract_protocol_ch1', 'label_ch1', 'label_protocol_ch1', 'taxid_ch1', 'hyb_protocol', 'scan_protocol', 'description', 'data_processing', 'platform_id', 'contact_name', 'contact_email', 'contact_phone', 'contact_fax', 'contact_department', 'contact_institute', 'contact_address', 'contact_city', 'contact_state', 'contact_zip/postal_code', 'contact_country', 'supplementary_file', 'data_row_count', 'relation', 'histopathological_diagnostic']

=== HISTOLOGY / GRADE / DIAGNOSIS ===

COLUMN: title
title
HF0017_U133P2      1
HF0024_U133P2      1
HF0026_U133P2      1
HF0031_U133P2      1
HF0050_U133P2      1
HF0066_U133P2      1
HF0087_U133P2      1
HF0088_U133P2      1
HF0089_U133P2      1
HF0108_U133P2      1
HF0120_U133P2      1
HF0131_U133P2      1
HF0137_U133P2      1
HF0138_U133P2      1
HF0141_U133P2      1


In [12]:
other_grade_pattern = (
    r"\bgrade\s*(?:i|ii|iii|1|2|3)\b|"
    r"\bwho\s*(?:grade\s*)?(?:i|ii|iii|1|2|3)\b"
)

# Lower-grade labels are evaluated within the GEO classification cell above.


In [13]:
# ---- B. TCGA-GBM and GTEx cortex (Toil) ------------------------------------------------------
ph = pd.read_csv(RAW / "TOIL" / "TcgaTargetGTEX_phenotype.txt.gz", sep="\t", compression="gzip", encoding="latin1")
c_id = ph.columns[0]
c_cat = find_col(ph, ["detailed_category", "detailed category"])
c_dis = find_col(ph, ["primary disease or tissue", "_primary_disease"])
c_typ = find_col(ph, ["_sample_type", "sample_type"])
c_std = find_col(ph, ["_study", "study"])
c_sex = find_col(ph, ["_gender", "gender"], required=False)

is_tcga = (ph[c_std].astype(str).str.upper() == "TCGA") & ph[c_dis].astype(str).str.contains("glioblastoma", case=False)
tcga_primary = is_tcga & (ph[c_typ].astype(str) == "Primary Tumor")
tcga_other = is_tcga & ~tcga_primary
is_gtex = (ph[c_std].astype(str).str.upper() == "GTEX") & ph[c_cat].astype(str).str.contains("brain", case=False) \
          & ph[c_cat].astype(str).str.contains("cortex", case=False)
print("TCGA GBM samples by type:\n", ph.loc[is_tcga, c_typ].value_counts().to_string())
print("GTEx cortex regions:\n", ph.loc[is_gtex, c_cat].value_counts().to_string())
for sid, t in zip(ph.loc[tcga_other, c_id], ph.loc[tcga_other, c_typ]):
    exclusions.append(("TCGA", sid, f"sample type {t} (not primary tumour)"))

toil = pd.concat([
    pd.DataFrame({"sample_id": ph.loc[tcga_primary, c_id], "cohort": "TCGA", "group": "GBM",
                  "region": "tumour", "sex": ph.loc[tcga_primary, c_sex] if c_sex else "unknown"}),
    pd.DataFrame({"sample_id": ph.loc[is_gtex, c_id], "cohort": "GTEx", "group": "control",
                  "region": ph.loc[is_gtex, c_cat].values, "sex": ph.loc[is_gtex, c_sex] if c_sex else "unknown"}),
], ignore_index=True)
require((toil.group == "GBM").sum() > 0 and (toil.group == "control").sum() > 0,
        "Toil phenotype: no TCGA-GBM primary tumours or no GTEx cortex samples were found. Check column names above.")
print(toil.groupby(["cohort", "group"]).size())

TCGA GBM samples by type:
 _sample_type
Primary Tumor          153
Recurrent Tumor         13
Solid Tissue Normal      5
GTEx cortex regions:
 detailed_category
Brain - Cortex                              105
Brain - Frontal Cortex (Ba9)                102
Brain - Anterior Cingulate Cortex (Ba24)     83
cohort  group  
GTEx    control    290
TCGA    GBM        153
dtype: int64


In [14]:
# C. TCGA clinical, optional phenotype, conservative IDH classification
surv_path = RAW / 'TCGA' / 'TCGA-GBM.survival.tsv.gz'
require_files([surv_path], 'TCGA survival required; run NB01.')
surv = pd.read_csv(surv_path, sep='\t', compression='gzip', low_memory=False)
s_id=find_col(surv,['sample']); s_t=find_col(surv,['OS.time','OS_time']); s_e=find_col(surv,['OS'])
surv = pd.DataFrame({'sample15':surv[s_id].astype(str).str.upper().str[:15],
    'os_time_days':pd.to_numeric(surv[s_t],errors='coerce'),
    'os_event':pd.to_numeric(surv[s_e],errors='coerce')})
surv = surv.drop_duplicates('sample15')
clin_path = RAW / 'TCGA' / 'TCGA-GBM.GDC_phenotype.tsv.gz'
clin2 = pd.DataFrame(columns=['sample15','age'])
if clin_path.exists():
    try:
        clin = pd.read_csv(clin_path,sep='\t',compression='gzip',low_memory=False)
        age_col=find_col(clin,['age_at_index.demographic','age_at_initial_pathologic_diagnosis','age_at_diagnosis.diagnoses'],required=False)
        clin2['sample15'] = clin.iloc[:,0].astype(str).str.upper().str[:15]
        if age_col:
            ages=pd.to_numeric(clin[age_col],errors='coerce')
            clin2['age']=ages.where(ages<=200,ages/365.25)
        else: clin2['age']=np.nan
        clin2=clin2.drop_duplicates('sample15')
    except Exception as exc:
        print('WARNING: TCGA phenotype unreadable:',exc)
        clin2=pd.DataFrame(columns=['sample15','age'])
else:
    print('WARNING: TCGA phenotype unavailable (earlier HTTP 403); age will be unknown')
tcga=toil.loc[toil.cohort=='TCGA'].copy()
tcga['sample15']=tcga.sample_id.astype(str).str.upper().str[:15]
n_before=len(tcga)
tcga=tcga.merge(surv,on='sample15',how='left',validate='many_to_one').merge(clin2,on='sample15',how='left',validate='many_to_one')
require(len(tcga)==n_before,'TCGA merge unexpectedly changed row count')
tcga['patient']=tcga.sample_id.astype(str).str.upper().str[:12]
mut_path=RAW/'TCGA'/'TCGA-GBM.mutect2_snv.tsv.gz'
# A mutation-only table cannot establish wildtype status (no coverage denominator).
tcga['idh_status']='unknown'
if mut_path.exists():
    try:
        mut=pd.read_csv(mut_path,sep='\t',compression='gzip',low_memory=False)
        m_s=find_col(mut,['Sample_ID','sample','Tumor_Sample_Barcode'])
        m_g=find_col(mut,['gene','Hugo_Symbol'])
        m_eff=find_col(mut,['effect','Variant_Classification'],required=False)
        is_idh=mut[m_g].astype(str).str.upper().isin(['IDH1','IDH2'])
        if m_eff:
            is_idh &= ~mut[m_eff].astype(str).str.contains('synonymous|intron|UTR|silent',case=False,regex=True)
        mutant_patients=set(mut.loc[is_idh,m_s].astype(str).str.upper().str[:12])
        tcga.loc[tcga.patient.isin(mutant_patients),'idh_status']='mutant'
    except Exception as exc:
        print('WARNING: mutation file could not be parsed:',exc)
else: print('WARNING: mutation file absent; IDH status unknown')
tcga['primary_recurrent']='primary'
print('TCGA survival matched:',int(tcga.os_time_days.notna().sum()),'of',len(tcga))
print('TCGA IDH status:',tcga.idh_status.value_counts().to_dict())


TCGA survival matched: 152 of 153
TCGA IDH status: {'unknown': 153}


In [15]:
# C2. Strict primary IDH-wildtype survival eligibility; unknown remains excluded
# Only verified wildtype cases can enter the primary survival cohort.
def tcga_reason(r):
    if r.idh_status == 'mutant': return 'IDH-mutant'
    if r.idh_status != 'wildtype': return 'IDH wildtype not verified'
    if pd.isna(r.age): return 'age unknown (adult status not verified)'
    if r.age < 18: return 'age < 18'
    if pd.isna(r.os_time_days) or pd.isna(r.os_event): return 'no overall-survival data'
    if r.os_time_days <= 0: return 'non-positive survival time'
    if r.os_event not in (0,1): return 'invalid survival event'
    return ''
tcga['survival_exclusion_reason']=tcga.apply(tcga_reason,axis=1)
tcga['included_survival']=tcga.survival_exclusion_reason.eq('')
for _,r in tcga.loc[~tcga.included_survival].iterrows():
    exclusions.append(('TCGA-survival',r.sample_id,r.survival_exclusion_reason))
# Expression comparison can include IDH-unknown primary tumors, flagged as such.
tcga['included']=tcga.idh_status.ne('mutant')
tcga['exclusion_reason']=np.where(tcga.included,'','IDH-mutant')
print('TCGA primary GBM:',len(tcga),'expression-eligible:',int(tcga.included.sum()),
      'strict survival-eligible:',int(tcga.included_survival.sum()))
if not tcga.included_survival.any():
    print('WARNING: No verified adult IDH-wildtype survival cohort. Obtain clinical age and validated IDH status before survival modeling.')
gtex=toil.loc[toil.cohort=='GTEx'].copy()
gtex['idh_status']='not_applicable';gtex['primary_recurrent']='not_applicable'
gtex['included']=True;gtex['exclusion_reason']=''


TCGA primary GBM: 153 expression-eligible: 153 strict survival-eligible: 0


In [16]:
# ---- D. CGGA clinical (full mode) ------------------------------------------------------------
cgga_sheets = {}
for tag in ["693", "325"]:
    cands = list((RAW / "CGGA" / f"extracted_{tag}").rglob("*clinical*.txt")) if (RAW / "CGGA").exists() else []
    if not cands:
        log(f"CGGA {tag}: clinical file not present (full mode needs it; see Troubleshooting).", "WARN")
        continue
    cl = pd.read_csv(cands[0], sep="\t", encoding="latin1")
    c_id = find_col(cl, ["CGGA_ID"]); c_prs = find_col(cl, ["PRS_type"]); c_gr = find_col(cl, ["Grade"])
    c_idh = find_col(cl, ["IDH_mutation_status"]); c_os = find_col(cl, ["OS"])
    c_cen = find_col(cl, ["Censor (alive=0; dead=1)", "Censor"]); c_age = find_col(cl, ["Age"])
    c_sex = find_col(cl, ["Gender"]); c_hist = find_col(cl, ["Histology"], required=False)
    c_mgmt = find_col(cl, ["MGMTp_methylation_status"], required=False)
    c_rad = find_col(cl, ["Radio_status (treated=1;un-treated=0)", "Radio_status"], required=False)
    c_chem = find_col(cl, ["Chemo_status (TMZ treated=1;un-treated=0)", "Chemo_status"], required=False)
    sh = pd.DataFrame({"sample_id": cl[c_id], "cohort": f"CGGA{tag}", "group": "GBM",
                       "primary_recurrent": cl[c_prs].astype(str).str.lower(),
                       "grade": cl[c_gr].astype(str), "histology": cl[c_hist] if c_hist else "",
                       "idh_status": cl[c_idh].astype(str).str.lower(),
                       "age": pd.to_numeric(cl[c_age], errors="coerce"), "sex": cl[c_sex].astype(str).str.lower(),
                       "os_time_days": pd.to_numeric(cl[c_os], errors="coerce"),
                       "os_event": pd.to_numeric(cl[c_cen], errors="coerce"),
                       "mgmt": cl[c_mgmt].astype(str).str.lower() if c_mgmt else "unknown",
                       "radio": pd.to_numeric(cl[c_rad], errors="coerce") if c_rad else np.nan,
                       "chemo": pd.to_numeric(cl[c_chem], errors="coerce") if c_chem else np.nan})
    print(f"\nCGGA{tag}: all glioma n={len(sh)}; grade/IDH/primary table:")
    print(pd.crosstab([sh.grade, sh.idh_status], sh.primary_recurrent).to_string())
    def reason(r):
        if not re.search(r"(?:^|\b)(?:iv|4)(?:$|\b)", r.grade.lower()):
            return "not WHO grade IV"
        if r.primary_recurrent != "primary":
            return f"{r.primary_recurrent} (not primary)"
        if "wild" not in r.idh_status:
            return f"IDH {r.idh_status}"
        if pd.isna(r.os_time_days) or pd.isna(r.os_event) or r.os_time_days <= 0:
            return "missing or invalid survival"
        if pd.notna(r.age) and r.age < 18:
            return "age < 18"
        return ""
    sh["exclusion_reason"] = sh.apply(reason, axis=1)
    sh["included"] = sh.exclusion_reason == ""
    for _, r in sh[~sh.included].iterrows():
        exclusions.append((f"CGGA{tag}", r.sample_id, r.exclusion_reason))
    cgga_sheets[tag] = sh
    print(f"CGGA{tag}: included primary IDH-wildtype GBM with survival = {int(sh.included.sum())}")


CGGA693: all glioma n=693; grade/IDH/primary table:
primary_recurrent   primary  recurrent
grade   idh_status                    
WHO II  mutant           96         39
        nan              12          4
        wildtype         30          7
WHO III mutant           88         83
        nan              20          5
        wildtype         36         23
WHO IV  mutant           24         25
        nan               7          3
        wildtype        109         81
nan     mutant            0          1
CGGA693: included primary IDH-wildtype GBM with survival = 104

CGGA325: all glioma n=325; grade/IDH/primary table:
primary_recurrent   nan  primary  recurrent  secondary
grade   idh_status                                    
WHO II  mutant        0       83          7          0
        nan           0        1          0          0
        wildtype      0       10          2          0
WHO III mutant        0       22         21          0
        wildtype      0       28 

In [17]:
# ---- E. duplicate-patient screens (metadata level; expression-level screen is repeated in NB03) --------
if len(cgga_sheets) == 2:
    a, b = cgga_sheets["693"], cgga_sheets["325"]
    shared_ids = set(a.sample_id) & set(b.sample_id)
    key = lambda d: d.dropna(subset=["age", "os_time_days"]).assign(
        k=lambda x: x.age.astype(int).astype(str) + "|" + x.sex + "|" + x.os_time_days.astype(int).astype(str))
    ka, kb = key(a), key(b)
    shared_key = set(ka.k) & set(kb.k)
    print(f"CGGA 693 vs 325: shared CGGA_IDs = {len(shared_ids)}; matching age|sex|OS triplets = {len(shared_key)}")
    manifest_update("overlap_screen", "cgga_693_vs_325", {"shared_ids": len(shared_ids), "matching_age_sex_os": len(shared_key)})
    if shared_ids or shared_key:
        log("Possible patient overlap between CGGA batches: NB03 will test expression correlation; "
            "overlapping patients will be removed from the second batch.", "WARN")

dup = []
if len(array_sheets) == 2:
    t1 = set(array_sheets["GSE4290"].sample_id); t2 = set(array_sheets["GSE50161"].sample_id)
    print("GEO accession overlap between GSE4290 and GSE50161:", len(t1 & t2))
    manifest_update("overlap_screen", "geo_array_accession_overlap", len(t1 & t2))

CGGA 693 vs 325: shared CGGA_IDs = 0; matching age|sex|OS triplets = 1
2026-10-20T07:48:54Z [WARN] Possible patient overlap between CGGA batches: NB03 will test expression correlation; overlapping patients will be removed from the second batch.


In [18]:
# ---- F. save sample sheets, exclusion log, cohort flow -----------------------------------------
sheets = {**array_sheets, "TCGA": tcga, "GTEx": gtex}
for tag, sh in cgga_sheets.items():
    sheets[f"CGGA{tag}"] = sh
for name, sh in sheets.items():
    sh.to_csv(P("meta", f"{name}_samples.csv"), index=False)
ex = pd.DataFrame(exclusions, columns=["cohort", "sample_id", "reason"])
ex.to_csv(P("meta", "exclusion_log.csv"), index=False)
flow = []
for name, sh in sheets.items():
    flow.append({"cohort": name, "n_listed": len(sh), "n_included": int(sh.get("included", pd.Series(True, index=sh.index)).sum()),
                 "n_gbm": int((sh.group == "GBM").sum()), "n_control": int((sh.group == "control").sum())})
flow = pd.DataFrame(flow)
flow.to_csv(P("meta", "cohort_flow.csv"), index=False)
print(flow.to_string(index=False))
print("\nExclusion reasons:\n", ex.groupby(["cohort", "reason"]).size().to_string() if len(ex) else "none")

 cohort  n_listed  n_included  n_gbm  n_control
   TCGA       153         153    153          0
   GTEx       290         290      0        290
CGGA693       693         104    693          0
CGGA325       325          73    325          0

Exclusion reasons:
 cohort         reason                                              
CGGA325        IDH mutant                                               11
               age < 18                                                  1
               not WHO grade IV                                        186
               recurrent (not primary)                                  24
               secondary (not primary)                                  30
CGGA693        IDH mutant                                               24
               IDH nan                                                   7
               age < 18                                                  1
               missing or invalid survival                             

In [19]:
# ---- G. metadata QC: missingness ---------------------------------------------------------------
rows = []
for name, sh in sheets.items():
    for c in ["age", "sex", "os_time_days", "os_event", "idh_status"]:
        if c in sh.columns:
            miss = sh[c].isna() | sh[c].astype(str).isin(["", "unknown", "nan", "not_annotated"])
            rows.append({"cohort": name, "field": c, "n_missing_or_unknown": int(miss.sum()), "n": len(sh)})
q = pd.DataFrame(rows)
q.to_csv(P("qc", "metadata_missingness.csv"), index=False)
print(q.pivot(index="cohort", columns="field", values="n_missing_or_unknown").to_string())
manifest_update("metadata", "sample_sheets", {k: str(P("meta", f"{k}_samples.csv", mkdir=False)) for k in sheets})
print("\nNB02 complete. Next: NB03 (preprocessing).")

field      age  idh_status  os_event  os_time_days  sex
cohort                                                 
CGGA325    0.0         1.0       9.0          12.0  0.0
CGGA693    1.0        51.0      30.0          36.0  0.0
GTEx       NaN         0.0       NaN           NaN  0.0
TCGA     153.0       153.0       1.0           1.0  1.0

NB02 complete. Next: NB03 (preprocessing).
